In [1]:
! which python

'which' is not recognized as an internal or external command,
operable program or batch file.


# Batch Normalization
One way to make deep networks easier to train is to use more sophisticated optimization procedures such as SGD+momentum, RMSProp, or Adam. Another strategy is to change the architecture of the network to make it easier to train. One idea along these lines is batch normalization which was recently proposed by [3].

The idea is relatively straightforward. Machine learning methods tend to work better when their input data consists of uncorrelated features with zero mean and unit variance. When training a neural network, we can preprocess the data before feeding it to the network to explicitly decorrelate its features; this will ensure that the first layer of the network sees data that follows a nice distribution. However even if we preprocess the input data, the activations at deeper layers of the network will likely no longer be decorrelated and will no longer have zero mean or unit variance since they are output from earlier layers in the network. Even worse, during the training process the distribution of features at each layer of the network will shift as the weights of each layer are updated.

The authors of [3] hypothesize that the shifting distribution of features inside deep neural networks may make training deep networks more difficult. To overcome this problem, [3] proposes to insert batch normalization layers into the network. At training time, a batch normalization layer uses a minibatch of data to estimate the mean and standard deviation of each feature. These estimated means and standard deviations are then used to center and normalize the features of the minibatch. A running average of these means and standard deviations is kept during training, and at test time these running averages are used to center and normalize features.

It is possible that this normalization strategy could reduce the representational power of the network, since it may sometimes be optimal for certain layers to have features that are not zero-mean or unit variance. To this end, the batch normalization layer includes learnable shift and scale parameters for each feature dimension.

[3] Sergey Ioffe and Christian Szegedy, "Batch Normalization: Accelerating Deep Network Training by Reducing
Internal Covariate Shift", ICML 2015.

**NB! For this task you need to copy `fc_net.py` from previous homework to current directory.**

In [2]:
import torch
from bn_layers import *
from fc_net import *
from data_utils import get_CIFAR10_data
from gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from solver import Solver

import time

import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# for auto-reloading external modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

def rel_error(x, y):

    """ returns relative error """

    x = x.to(torch.float64)
    y = y.to(torch.float64)
    
    return torch.max(torch.abs(x - y) / (torch.max(torch.tensor(1e-8, dtype=torch.float64), torch.abs(x) + torch.abs(y))))

In [3]:
# set default tensor type
torch.set_default_dtype(torch.float64)

# Load the (preprocessed) CIFAR10 data.
data = get_CIFAR10_data('datasets/cifar-10-batches-py')

for k, v in list(data.items()):
    
    data[k] = torch.tensor(v, dtype=torch.float64)
    print('%s: ' % k, data[k].shape)

d:\Artem\Tartu_CS\1st year\Neural Nets\HW6\data_utils.py:15: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return  pickle.load(f, encoding='latin1')


X_train:  torch.Size([49000, 3, 32, 32])
y_train:  torch.Size([49000])
X_val:  torch.Size([1000, 3, 32, 32])
y_val:  torch.Size([1000])
X_test:  torch.Size([1000, 3, 32, 32])
y_test:  torch.Size([1000])


## Batch normalization: Forward

**Task 6.8**

In the file `bn_layers.py`, implement the batch normalization forward pass in the function `batchnorm_forward`. Once you have done so, run the following to test your implementation.

In [4]:
# Check the training-time forward pass by checking means and variances
# of features both before and after batch normalization

# Simulate the forward pass for a two-layer network

torch.manual_seed(231)

N, D1, D2, D3 = 200, 50, 60, 3

X = torch.randn(N, D1)
W1 = torch.randn(D1, D2)
W2 = torch.randn(D2, D3)
a = torch.relu(X @ W1) @ W2

print('Before batch normalization:')
print('  means: ', a.mean(dim=0).cpu().numpy())
print('  stds: ', a.std(dim=0, unbiased=False).cpu().numpy())

# Means should be close to zero and stds close to one
print('After batch normalization (gamma=1, beta=0)')
gamma = torch.ones(D3)
beta = torch.zeros(D3)
a_norm, _ = batchnorm_forward(a, gamma, beta, {'mode': 'train'})
print('  means: ', a_norm.mean(dim=0).cpu().numpy())
print('  std: ', a_norm.std(dim=0, unbiased=False).cpu().numpy())

print()
assert torch.allclose(a_norm.mean(dim=0), torch.zeros(D3), atol=1e-7), 'Task 6.8 failed: means not close to zero' 

# Now means should be close to beta and stds close to gamma
gamma = torch.tensor([1.0, 2.0, 3.0])
beta = torch.tensor([11.0, 12.0, 13.0])
a_norm, _ = batchnorm_forward(a, gamma, beta, {'mode': 'train'})
print('After batch normalization (nontrivial gamma, beta)')
print('  means: ', a_norm.mean(dim=0).cpu().numpy())
print('  stds: ', a_norm.std(dim=0, unbiased=False).cpu().numpy())

print()
assert torch.allclose(a_norm.mean(dim=0), beta, atol=1e-7), 'Task 6.8 failed: means not close to beta' 
assert torch.allclose(a_norm.std(dim=0, unbiased=False), gamma, atol=1e-7), 'Task 6.8 failed: stds not close to gamma'

print('Task 6.8 passed!')

Before batch normalization:
  means:  [-32.22324013  14.01781659  -6.46534246]
  stds:  [31.36188852 27.11310322 37.94029312]
After batch normalization (gamma=1, beta=0)
  means:  [1.66533454e-16 7.77156117e-18 3.55271368e-17]
  std:  [0.99999999 0.99999999 1.        ]

After batch normalization (nontrivial gamma, beta)
  means:  [11. 12. 13.]
  stds:  [0.99999999 1.99999999 2.99999999]

Task 6.8 passed!


In [5]:
# Check the test-time forward pass by running the training-time
# forward pass many times to warm up the running averages, and then
# checking the means and variances of activations after a test-time
# forward pass.

torch.manual_seed(231)
N, D1, D2, D3 = 200, 50, 60, 3
W1 = torch.randn(D1, D2)
W2 = torch.randn(D2, D3)

bn_param = {'mode': 'train'}

gamma = torch.ones(D3)
beta = torch.zeros(D3)

for t in range(50):
    X = torch.randn(N, D1)
    a = torch.relu(X @ W1) @ W2
    batchnorm_forward(a, gamma, beta, bn_param)

bn_param['mode'] = 'test'
X = torch.randn(N, D1)
a = torch.relu(X @ W1) @ W2
a_norm, _ = batchnorm_forward(a, gamma, beta, bn_param)

# Means should be close to zero and stds close to one, but will be
# noisier than training-time forward passes.
print('After batch normalization (test-time):')
print('  means: ', a_norm.mean(dim=0).cpu().numpy())
print('  stds: ', a_norm.std(dim=0, unbiased=False).cpu().numpy())

After batch normalization (test-time):
  means:  [ 0.13767817 -0.00193477  0.02051705]
  stds:  [0.94535865 0.95542323 0.93944188]


## Batch Normalization: backward

**Task 6.9**

Now implement the backward pass for batch normalization in the function `batchnorm_backward`.

To derive the backward pass you should write out the computation graph for batch normalization and backprop through each of the intermediate nodes. Some intermediates may have multiple outgoing branches; make sure to sum gradients across these branches in the backward pass.

Once you have finished, run the following to numerically check your backward pass.

In [6]:
# Gradient check batchnorm backward pass

def fx(x_):
    out, _ = batchnorm_forward(x_, gamma, beta, dict(bn_param))
    return out

def fg(gamma_):
    out, _ = batchnorm_forward(x, gamma_, beta, dict(bn_param))
    return out

def fb(beta_):
    out, _ = batchnorm_forward(x, gamma, beta_, dict(bn_param))
    return out

torch.manual_seed(231)
N, D = 4, 5
x = 5 * torch.randn(N, D) + 12
gamma = torch.randn(D)
beta = torch.randn(D)
dout = torch.randn(N, D)

bn_param = {'mode': 'train'}

dx_num = eval_numerical_gradient_array(fx, x, dout)
dgamma_num = eval_numerical_gradient_array(fg, gamma, dout)
dbeta_num = eval_numerical_gradient_array(fb, beta, dout)

_, cache = batchnorm_forward(x, gamma, beta, dict(bn_param))
dx, dgamma, dbeta = batchnorm_backward(dout, cache)

dx_diff = rel_error(dx_num, dx)
dgamma_diff = rel_error(dgamma_num, dgamma)
dbeta_diff = rel_error(dbeta_num, dbeta)

print()
assert dx_diff < 1e-7, f'Task 6.9 failed: dx error too high ({dx_diff:.6e})'
assert dgamma_diff < 1e-7, f'Task 6.9 failed: dgamma error too high ({dgamma_diff:.6e})'
assert dbeta_diff < 1e-7, f'Task 6.9 failed: dbeta error too high ({dbeta_diff:.6e})'

print('Task 6.9 passed!')
print('dx error: ', rel_error(dx_num, dx))
print('dgamma error: ', rel_error(dgamma_num, dgamma))
print('dbeta error: ', rel_error(dbeta_num, dbeta))


Task 6.9 passed!
dx error:  tensor(1.2254e-09)
dgamma error:  tensor(9.6687e-12)
dbeta error:  tensor(3.5423e-12)


## Batch Normalization: alternative backward

**Task 6.10**

In class we talked about two different implementations for the sigmoid backward pass. One strategy is to write out a computation graph composed of simple operations and backprop through all intermediate values. Another strategy is to work out the derivatives on paper. For the sigmoid function, it turns out that you can derive a very simple formula for the backward pass by simplifying gradients on paper.

Surprisingly, it turns out that you can also derive a simple expression for the batch normalization backward pass if you work out derivatives on paper and simplify. After doing so, implement the simplified batch normalization backward pass in the function `batchnorm_backward_alt` and compare the two implementations by running the following. Your two implementations should compute nearly identical results, but the alternative implementation should be a bit faster.

In [7]:
torch.manual_seed(231)
N, D = 100, 500
x = 5 * torch.randn(N, D) + 12
gamma = torch.randn(D)
beta = torch.randn(D)
dout = torch.randn(N, D)

bn_param = {'mode': 'train'}
out, cache = batchnorm_forward(x, gamma, beta, dict(bn_param))

t1 = time.time()
dx1, dgamma1, dbeta1 = batchnorm_backward(dout, cache)
t2 = time.time()
dx2, dgamma2, dbeta2 = batchnorm_backward_alt(dout, cache)
t3 = time.time()

dx_diff = rel_error(dx1, dx2)
dgamma_diff = rel_error(dgamma1, dgamma2)
dbeta_diff = rel_error(dbeta1, dbeta2)

print()
assert dx_diff < 1e-7, f'Task 6.10 failed: dx difference too high ({dx_diff:.6e})'
assert dgamma_diff < 1e-7, f'Task 6.10 failed: dgamma difference too high ({dgamma_diff:.6e})'
assert dbeta_diff < 1e-7, f'Task 6.10 failed: dbeta difference too high ({dbeta_diff:.6e})'

print('Task 6.10 passed!')
print('dx difference: ', rel_error(dx1, dx2))
print('dgamma difference: ', rel_error(dgamma1, dgamma2))
print('dbeta difference: ', rel_error(dbeta1, dbeta2))
print('speedup: %.2fx' % ((t2 - t1) / (t3 - t2)))

AttributeError: 'NoneType' object has no attribute 'to'

## Fully Connected Nets with Batch Normalization

**Task 6.11**

Now that you have a working implementation for batch normalization, go back to your `FullyConnectedNet` in the file `fc_net.py`. Modify your implementation to add batch normalization.

Concretely, when the flag `use_batchnorm` is `True` in the constructor, you should insert a batch normalization layer before each ReLU nonlinearity. The outputs from the last layer of the network should not be normalized. Once you are done, run the following to gradient-check your implementation.

HINT: You might find it useful to use a helper layer from `layer_utils.py`.

In [ ]:
torch.manual_seed(231)

N, D, H1, H2, C = 2, 15, 20, 30, 10
X = torch.randn(N, D, dtype=torch.float64)
y = torch.randint(low=0, high=C, size=(N,), dtype=torch.long)

for reg in [0, 3.14]:
    print('Running check with reg = ', reg)
    model = FullyConnectedNet(
        [H1, H2], input_dim=D, num_classes=C,
        reg=reg, weight_scale=5e-2, dtype=torch.float64,
        use_batchnorm=True
    )

    loss, grads = model.loss(X, y)
    print('Initial loss: ', loss.item())

    for name in sorted(grads):
        f = lambda W: model.loss(X, y)[0]
        grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
        print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))
    if reg == 0:
        print()

    assert rel_error(grad_num, grads[name]) < 1e-7, f'Task 6.11 failed: {name} gradient error too high ({rel_error(grad_num, grads[name]):.6e})'

print()
print('Task 6.11 passed!')

# Batchnorm for deep networks
Run the following to train a six-layer network on a subset of 1000 training examples both with and without batch normalization.

In [ ]:
torch.manual_seed(231)  # For reproducibility

# Try training a very deep net with batchnorm
hidden_dims = [100, 100, 100, 100, 100]

num_train = 1000
small_data = {
    'X_train': data['X_train'][:num_train],
    'y_train': data['y_train'][:num_train],
    'X_val': data['X_val'],
    'y_val': data['y_val'],
}

weight_scale = 2e-2

print("With batch normalization:")
bn_model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, use_batchnorm=True)
bn_solver = Solver(
    bn_model, small_data,
    num_epochs=10, batch_size=50,
    update_rule='adam',
    optim_config={
        'learning_rate': 1e-3,
    },
    verbose=True, print_every=200
)
bn_solver.train()

print("Without batch normalization:")
model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, use_batchnorm=False)
solver = Solver(
    model, small_data,
    num_epochs=10, batch_size=50,
    update_rule='adam',
    optim_config={
        'learning_rate': 1e-3,
    },
    verbose=True, print_every=200
)
solver.train()

Run the following to visualize the results from two networks trained above. You should find that using batch normalization helps the network to converge much faster.

In [ ]:
plt.subplot(3, 1, 1)
plt.title('Training loss')
plt.xlabel('Iteration')

plt.subplot(3, 1, 2)
plt.title('Training accuracy')
plt.xlabel('Epoch')

plt.subplot(3, 1, 3)
plt.title('Validation accuracy')
plt.xlabel('Epoch')

plt.subplot(3, 1, 1)
plt.plot(solver.loss_history, 'o', label='baseline')
plt.plot(bn_solver.loss_history, 'o', label='batchnorm')

plt.subplot(3, 1, 2)
plt.plot(solver.train_acc_history, '-o', label='baseline')
plt.plot(bn_solver.train_acc_history, '-o', label='batchnorm')

plt.subplot(3, 1, 3)
plt.plot(solver.val_acc_history, '-o', label='baseline')
plt.plot(bn_solver.val_acc_history, '-o', label='batchnorm')
  
for i in [1, 2, 3]:
  plt.subplot(3, 1, i)
  plt.legend(loc='upper center', ncol=4)
plt.gcf().set_size_inches(15, 15)
plt.show()

# Batch normalization and initialization
We will now run a small experiment to study the interaction of batch normalization and weight initialization.

The first cell will train 8-layer networks both with and without batch normalization using different scales for weight initialization. The second layer will plot training accuracy, validation set accuracy, and training loss as a function of the weight initialization scale.

In [ ]:
torch.manual_seed(231)
# Try training a very deep net with batchnorm
hidden_dims = [50, 50, 50, 50, 50, 50, 50]

num_train = 1000
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

bn_solvers = {}
solvers = {}
weight_scales = torch.logspace(-4, 0, steps=20)
for i, weight_scale in enumerate(weight_scales):
  print('Running weight scale %d / %d' % (i + 1, len(weight_scales)))
  bn_model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, use_batchnorm=True)
  model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, use_batchnorm=False)

  bn_solver = Solver(bn_model, small_data,
                  num_epochs=10, batch_size=50,
                  update_rule='adam',
                  optim_config={
                    'learning_rate': 1e-3,
                  },
                  verbose=False, print_every=200)
  bn_solver.train()
  bn_solvers[weight_scale] = bn_solver

  solver = Solver(model, small_data,
                  num_epochs=10, batch_size=50,
                  update_rule='adam',
                  optim_config={
                    'learning_rate': 1e-3,
                  },
                  verbose=False, print_every=200)
  solver.train()
  solvers[weight_scale] = solver

In [ ]:
# Plot results of weight scale experiment
best_train_accs, bn_best_train_accs = [], []
best_val_accs, bn_best_val_accs = [], []
final_train_loss, bn_final_train_loss = [], []

solvers = {float(k.item()): v for k, v in solvers.items()}
bn_solvers = {float(k.item()): v for k, v in bn_solvers.items()}
weight_scales = [float(ws.item()) for ws in weight_scales]

for ws in weight_scales:
  best_train_accs.append(max(solvers[ws].train_acc_history))
  bn_best_train_accs.append(max(bn_solvers[ws].train_acc_history))
  
  best_val_accs.append(max(solvers[ws].val_acc_history))
  bn_best_val_accs.append(max(bn_solvers[ws].val_acc_history))
  
  final_train_loss.append(torch.mean(torch.tensor(solvers[ws].loss_history[-100:])).item())
  bn_final_train_loss.append(torch.mean(torch.tensor(bn_solvers[ws].loss_history[-100:])).item())

  
plt.subplot(3, 1, 1)
plt.title('Best val accuracy vs weight initialization scale')
plt.xlabel('Weight initialization scale')
plt.ylabel('Best val accuracy')
plt.semilogx(weight_scales, best_val_accs, '-o', label='baseline')
plt.semilogx(weight_scales, bn_best_val_accs, '-o', label='batchnorm')
plt.legend(ncol=2, loc='lower right')

plt.subplot(3, 1, 2)
plt.title('Best train accuracy vs weight initialization scale')
plt.xlabel('Weight initialization scale')
plt.ylabel('Best training accuracy')
plt.semilogx(weight_scales, best_train_accs, '-o', label='baseline')
plt.semilogx(weight_scales, bn_best_train_accs, '-o', label='batchnorm')
plt.legend()

plt.subplot(3, 1, 3)
plt.title('Final training loss vs weight initialization scale')
plt.xlabel('Weight initialization scale')
plt.ylabel('Final training loss')
plt.semilogx(weight_scales, final_train_loss, '-o', label='baseline')
plt.semilogx(weight_scales, bn_final_train_loss, '-o', label='batchnorm')
plt.legend()
plt.gca().set_ylim(1.0, 3.5)

plt.gcf().set_size_inches(10, 15)
plt.show()

**Task 6.12**

### Question:
Does behaviour of batch normalization change if you apply it before activation function or after?

### Answer:


# Spatial Batch Normalization
We already saw that batch normalization is a very useful technique for training deep fully-connected networks. Batch normalization can also be used for convolutional networks, but we need to tweak it a bit; the modification will be called "spatial batch normalization."

Normally batch-normalization accepts inputs of shape `(N, D)` and produces outputs of shape `(N, D)`, where we normalize across the minibatch dimension `N`. For data coming from convolutional layers, batch normalization needs to accept inputs of shape `(N, C, H, W)` and produce outputs of shape `(N, C, H, W)` where the `N` dimension gives the minibatch size and the `(H, W)` dimensions give the spatial size of the feature map.

If the feature map was produced using convolutions, then we expect the statistics of each feature channel to be relatively consistent both between different imagesand different locations within the same image. Therefore spatial batch normalization computes a mean and variance for each of the `C` feature channels by computing statistics over both the minibatch dimension `N` and the spatial dimensions `H` and `W`.

## Spatial batch normalization: forward

**Task 6.13**

In the file `bn_layers.py`, implement the forward pass for spatial batch normalization in the function `spatial_batchnorm_forward`. Check your implementation by running the following:

In [ ]:
torch.manual_seed(231)
# Check the training-time forward pass by checking means and variances
# of features both before and after spatial batch normalization

N, C, H, W = 2, 3, 4, 5
x = 4 * torch.randn(N, C, H, W) + 10

print('Before spatial batch normalization:')
print('  Shape: ', x.shape)
print('  Means: ', x.mean(dim=(0, 2, 3)))
print('  Stds: ', x.std(dim=(0, 2, 3)))

# Means should be close to zero and stds close to one
gamma, beta = torch.ones(C), torch.zeros(C)
bn_param = {'mode': 'train'}
out, _ = spatial_batchnorm_forward(x, gamma, beta, bn_param)
print('After spatial batch normalization:')
print('  Shape: ', out.shape)
print('  Means: ', out.mean(dim=(0, 2, 3)))
print('  Stds: ', out.std(dim=(0, 2, 3)))

# Means should be close to beta and stds close to gamma
gamma, beta = torch.tensor([3, 4, 5]), torch.tensor([6, 7, 8])
out, _ = spatial_batchnorm_forward(x, gamma, beta, bn_param)
print('After spatial batch normalization (nontrivial gamma, beta):')
print('  Shape: ', out.shape)
print('  Means: ', out.mean(dim=(0, 2, 3)))
print('  Stds: ', out.std(dim=(0, 2, 3)))

print('Task 6.13 passed!')

In [ ]:
torch.manual_seed(231)
# Check the test-time forward pass by running the training-time
# forward pass many times to warm up the running averages, and then
# checking the means and variances of activations after a test-time
# forward pass.
N, C, H, W = 10, 4, 11, 12

bn_param = {'mode': 'train'}
gamma = torch.ones(C)
beta = torch.zeros(C)

for t in range(50):
  x = 2.3 * torch.randn(N, C, H, W) + 13
  spatial_batchnorm_forward(x, gamma, beta, bn_param)

bn_param['mode'] = 'test'
x = 2.3 * torch.randn(N, C, H, W) + 13
a_norm, _ = spatial_batchnorm_forward(x, gamma, beta, bn_param)

# Means should be close to zero and stds close to one, but will be
# noisier than training-time forward passes.
print('After spatial batch normalization (test-time):')
print('  means: ', a_norm.mean(dim=(0, 2, 3)))
print('  stds: ', a_norm.std(dim=(0, 2, 3)))

## Spatial batch normalization: backward

**Task 6.14**

In the file `bn_layers.py`, implement the backward pass for spatial batch normalization in the function `spatial_batchnorm_backward`. Run the following to check your implementation using a numeric gradient check:

In [ ]:
torch.manual_seed(231)

N, C, H, W = 2, 3, 4, 5

x = 5 * torch.randn(N, C, H, W) + 12
gamma = torch.randn(C)
beta = torch.randn(C)
dout = torch.randn(N, C, H, W)

bn_param = {'mode': 'train'}
fx = lambda x: spatial_batchnorm_forward(x, gamma, beta, bn_param)[0]
fg = lambda a: spatial_batchnorm_forward(x, gamma, beta, bn_param)[0]
fb = lambda b: spatial_batchnorm_forward(x, gamma, beta, bn_param)[0]

dx_num = eval_numerical_gradient_array(fx, x, dout)
da_num = eval_numerical_gradient_array(fg, gamma, dout)
db_num = eval_numerical_gradient_array(fb, beta, dout)

_, cache = spatial_batchnorm_forward(x, gamma, beta, bn_param)
dx, dgamma, dbeta = spatial_batchnorm_backward(dout, cache)

dx_diff = rel_error(dx_num, dx)
dgamma_diff = rel_error(da_num, dgamma)
dbeta_diff = rel_error(db_num, dbeta)

print()
assert dx_diff < 1e-7, f'Task 6.14 failed: dx error too high ({dx_diff:.6e})'
assert dgamma_diff < 1e-7, f'Task 6.14 failed: dgamma error too high ({dgamma_diff:.6e})'
assert dbeta_diff < 1e-7, f'Task 6.14 failed: dbeta error too high ({dbeta_diff:.6e})'

print('Task 6.14 passed!')
print('dx error: ', rel_error(dx_num, dx))
print('dgamma error: ', rel_error(da_num, dgamma))
print('dbeta error: ', rel_error(db_num, dbeta))